<a href="https://colab.research.google.com/github/rajaonsonella/crosstalk-uoft/blob/main/notebooks/4_0_TabPFN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TabPFN

# Set up

⚙️ Step 1: Set your notebook to GPU

The next two cells take ~2 min.... start running them now while we talk! 👇👇

In [1]:
# get workshop code
import os

IN_COLAB = os.getenv("COLAB_RELEASE_TAG")
if IN_COLAB:
    !git clone https://github.com/rajaonsonella/crosstalk-uoft
    !pip install -r crosstalk-uoft/requirements.txt

!pip install tabpfn tabpfn-client gdown pyarrow

Cloning into 'crosstalk-uoft'...
remote: Enumerating objects: 452, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 452 (delta 71), reused 39 (delta 37), pack-reused 347 (from 2)
Receiving objects: 100% (452/452), 47.74 MiB | 18.55 MiB/s, done.
Resolving deltas: 100% (245/245), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.8/165.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 60.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of requests[socks] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.2/753.2 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.3/279.3 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
  Attempting unin

# Imports

In [1]:
# Core
import os
import sys

if os.getenv("COLAB_RELEASE_TAG"):
    sys.path.append('./crosstalk-uoft')
else:
    sys.path.append('..')

# Numerical & Datascience & Vis
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Chemistry
import dataset

# ML
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from tabpfn import TabPFNClassifier

In [3]:
# Download data from google drive
import gdown

file_ids = {'test_inputs' : '15iMvnmIraM-geCI-vG9iR5naliWfh5tA',
            'train': '11S5p0QgP1X9rOFiIjNSLydLenJwm7hle'}

for name, file_id in file_ids.items():
    filename = f'crosstalk_{name}.parquet'
    if not os.path.exists(filename):
        gdown.download(id=file_id, output=filename, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=15iMvnmIraM-geCI-vG9iR5naliWfh5tA
From (redirected): https://drive.google.com/uc?id=15iMvnmIraM-geCI-vG9iR5naliWfh5tA&confirm=t&uuid=af49e72c-0838-4de1-881c-95047babde07
To: /content/crosstalk_test_inputs.parquet
100%|██████████| 1.52G/1.52G [00:27<00:00, 55.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=11S5p0QgP1X9rOFiIjNSLydLenJwm7hle
From (redirected): https://drive.google.com/uc?id=11S5p0QgP1X9rOFiIjNSLydLenJwm7hle&confirm=t&uuid=7175d935-6aad-4212-807f-cd1508942eae
To: /content/crosstalk_train.parquet
100%|██████████| 1.97G/1.97G [00:52<00:00, 37.8MB/s]


## 📈 Load the data + features

TabPFN works best on datasets within its recommended size limits. The current default (TabPFN-3) supports up to 1,000,000 × 200, 100,000 × 2,000, or 1,000 × 20,000 (rows × features) — larger feature counts trade off against row capacity. See the Models page for the limits of other checkpoints.

In [ ]:
x = dataset.load_x("crosstalk_train.parquet", ["ECFP6"])
y = dataset.load_y("crosstalk_train.parquet")

FEAT_DIM = 250
og_dim = x.shape[1]
feat_mask = np.zeros(og_dim, dtype=bool)

selected_indices = np.random.choice(og_dim, FEAT_DIM, replace=False)
feat_mask[selected_indices] = True
x = x[:, feat_mask]

N_SAMPLES = 5000
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    train_size=N_SAMPLES,
    stratify=y,
    random_state=42
)

# Instanciate a model

https://github.com/PriorLabs/tabpfn

In [7]:
model = TabPFNClassifier()
model

TabPFNClassifier()

1. Open https://ux.priorlabs.ai in a browser and log in (or register)
  2. Accept the license on the Licenses tab
  3. Copy your API Key from https://ux.priorlabs.ai/account

In [9]:
import os; os.environ["TABPFN_TOKEN"] = ""

In [ ]:
model.fit(x_train, y_train)

y_pred = model.predict_proba(x_val)[:, 1]

auc = roc_auc_score(y_val, y_pred)
auprc = average_precision_score(y_val, y_pred)

print(f"AUROC: {auc:.4f}")
print(f"AUPRC: {auprc:.4f}")

tabpfn-v3-classifier-v3_default.ckpt:   0%|          | 0.00/213M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
